In [13]:
import io
import time
import random
import requests
import zipfile
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


def download_day(date, nodes=None, max_retries=5):
    """Download a single day of CAISO DAM LMP data with backoff."""
    base_url = "https://oasis.caiso.com/oasisapi/SingleZip"
    date_str = date.strftime("%Y%m%d")

    params = {
        "resultformat": "6",
        "queryname": "PRC_LMP",
        "version": "12",
        "market_run_id": "DAM",
        "startdatetime": f"{date_str}T08:00-0000",
        "enddatetime": f"{date_str}T08:00-0000",
        "grp_type": "ALL"
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(base_url, params=params, timeout=60)
            
            # Handle rate limit explicitly
            if response.status_code == 429:
                wait = 3 * (attempt + 1) + random.uniform(0, 2)
                print(f"⚠️ 429 Too Many Requests for {date_str}. Retrying in {wait:.1f}s...")
                time.sleep(wait)
                continue
            
            response.raise_for_status()
            
            # Try reading the zip file
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                for name in z.namelist():
                    if "DAM_LMP" in name and name.endswith(".csv"):
                        with z.open(name) as f:
                            df = pd.read_csv(f)
                            if nodes is not None:
                                df = df[df["NODE"].isin(nodes)]
                            return df

        except Exception as e:
            # Retry on any error
            wait = 2 * (attempt + 1) + random.uniform(0, 1.5)
            print(f"⚠️ Error for {date_str}: {e}. Retrying in {wait:.1f}s...")
            time.sleep(wait)

    print(f"❌ Failed after retries: {date_str}")
    return None


def fetch_caiso_dam_lmp_parallel(start_date, end_date, nodes=None, max_workers=3):
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    date_list = [start_dt + timedelta(days=i) for i in range((end_dt - start_dt).days + 1)]

    all_data = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(download_day, date, nodes): date
            for date in date_list
        }

        pbar = tqdm(total=len(futures), desc="CAISO Download")

        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                all_data.append(result)
            pbar.update(1)

        pbar.close()

    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


# Example use:
if __name__ == "__main__":
    df = fetch_caiso_dam_lmp_parallel(
        start_date="2024-10-05",
        end_date="2024-10-06",
        max_workers=3  # KEEP THIS LOW — CAISO rate limits heavily
    )

    print(df.head())
    df.to_csv("../Missing_CAISO/caiso_dam_lmp_10_5.csv", index=False)


CAISO Download:   0%|          | 0/2 [00:00<?, ?it/s]

⚠️ 429 Too Many Requests for 20241006. Retrying in 5.0s...


CAISO Download: 100%|██████████| 2/2 [01:12<00:00, 36.49s/it]


       INTERVALSTARTTIME_GMT        INTERVALENDTIME_GMT      OPR_DT  OPR_HR  \
0  2024-10-05T13:00:00-00:00  2024-10-05T14:00:00-00:00  2024-10-05       7   
1  2024-10-06T02:00:00-00:00  2024-10-06T03:00:00-00:00  2024-10-05      20   
2  2024-10-06T01:00:00-00:00  2024-10-06T02:00:00-00:00  2024-10-05      19   
3  2024-10-06T00:00:00-00:00  2024-10-06T01:00:00-00:00  2024-10-05      18   
4  2024-10-05T15:00:00-00:00  2024-10-05T16:00:00-00:00  2024-10-05       9   

   OPR_INTERVAL    NODE_ID_XML        NODE_ID           NODE MARKET_RUN_ID  \
0             0  0096WD_7_N001  0096WD_7_N001  0096WD_7_N001           DAM   
1             0  0096WD_7_N001  0096WD_7_N001  0096WD_7_N001           DAM   
2             0  0096WD_7_N001  0096WD_7_N001  0096WD_7_N001           DAM   
3             0  0096WD_7_N001  0096WD_7_N001  0096WD_7_N001           DAM   
4             0  0096WD_7_N001  0096WD_7_N001  0096WD_7_N001           DAM   

  LMP_TYPE XML_DATA_ITEM  PNODE_RESMRID GRP_TYPE  POS   